# Risco e retorno da empresa do seu grupo
**ED0139 · Finanças Corporativas · 2026-2 · Prof. Sérgio Cardoso · UFC**

Versão do aluno. Entrega pelo SIGAA até 16/09.

Este notebook refaz, em Python, exatamente a mesma conta da planilha. A regra da disciplina
é que os dois caminhos cheguem ao mesmo número: método diferente com resultado igual é a
defesa contra erro de fórmula e contra copiar sem entender.

O que você vai calcular:

1. o retorno diário da sua empresa a partir dos preços de fechamento;
2. o retorno médio e o desvio-padrão, ao dia e ao ano;
3. a covariância e a correlação com o BOVA11, o índice inteiro;
4. o risco de uma carteira que junta as duas coisas.

Os dados são do pacote congelado da disciplina: fechamentos oficiais da B3 entre
20/08/2021 e 20/08/2026, sem ajuste por proventos.

---

## Antes de começar, leia isto

Este notebook roda **na nuvem do Google**. Você não instala nada no seu computador.

**1. Salve uma cópia sua, agora.** Menu **Arquivo → Salvar uma cópia no Drive**.
Se você pular este passo, tudo o que escrever se perde ao fechar a aba.

**2. Para executar uma célula**, clique no botão ▶ que aparece à esquerda dela,
ou aperte **Shift + Enter**.

**3. Execute na ordem, de cima para baixo.** Uma célula pulada quebra todas as de
baixo, porque cada uma usa o resultado da anterior.

**4. Você só escreve onde estiver marcado** `>>> SUA VEZ`. O resto já está pronto e
não deve ser alterado.

**5. Se parar de responder ou você ficar muito tempo longe**, o Colab desconecta.
Não perdeu nada: use **Ambiente de execução → Executar tudo** e ele refaz tudo em
alguns segundos.

**6. Não precisa baixar nem enviar arquivo de dados.** Os preços são lidos direto
da internet.

---

## Passo 0 · Identificação

Preencha os três campos abaixo e execute a célula. Eles entram no arquivo que você
vai entregar.

In [ ]:
#@title Identificação { display-mode: "form" }
NOME = ""       #@param {type:"string"}
MATRICULA = ""  #@param {type:"string"}
GRUPO = ""      #@param {type:"string"}

print(f"{NOME or '(sem nome)'} · matrícula {MATRICULA or '(vazia)'} · grupo {GRUPO or '(vazio)'}")

## Passo 1 · Os dados

Os preços são lidos direto do pacote congelado da disciplina, pela internet. Não é
preciso baixar nem enviar arquivo nenhum. A única coisa que muda de um grupo para
outro é o ticker.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CAMINHO = "https://sergiocardoso.pro.br/ufc/financas-corporativas/2026-2/dados/precos/fechamentos_diarios_2021-2026_congelado_2026-08-24.csv"
TICKER  = "ABEV3"   # <<< o ticker do SEU grupo
PREGOES_ANO = 252

precos = pd.read_csv(CAMINHO, parse_dates=["data"]).set_index("data")
precos[[TICKER, "BOVA11"]].tail()

## Passo 1 · O cuidado que vem antes da conta

Antes de qualquer estatística, olhe os maiores movimentos da série e pergunte de cada um se
foi mercado ou se foi evento societário. Grupamento e desdobramento mudam a unidade do preço
sem mudar a riqueza de ninguém, e um único dia mal tratado contamina cinco anos de conta.

In [ ]:
variacao = precos[TICKER].pct_change()
variacao.abs().sort_values(ascending=False).head(5).map(lambda x: f"{x:.1%}")

**A Hapvida fez grupamento de 15 para 1 em 06/06/2025.** O grupo da HAPV3 precisa corrigir
os preços anteriores a essa data, multiplicando por 15. Os demais grupos deixam `FATOR = 1`,
porque nenhuma das outras sete empresas teve evento do tipo no período.

In [ ]:
# >>> SUA VEZ. Troque cada  ...  pela conta certa.
# >>> As dicas estão nos comentários ao lado.

DATA_EVENTO = "2025-06-06"
FATOR = 1           # HAPV3: 15. Demais empresas: 1.

serie = precos[TICKER].copy()
if FATOR != 1:
    serie.loc[:DATA_EVENTO] = ...          # corrija os preços anteriores ao evento
    serie.loc[DATA_EVENTO] = ...           # o próprio dia ex já vem agrupado

serie.tail()

## Passo 2 · Retorno

$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

O primeiro dia da série não tem retorno, porque não existe preço anterior. É por isso que
1.248 preços produzem 1.247 retornos.

In [ ]:
# >>> SUA VEZ. Troque cada  ...  pela conta certa.
# >>> As dicas estão nos comentários ao lado.

retorno = ...              # dica: pct_change() e depois dropna()
retorno_ibov = ...

print(f"{len(serie)} preços produziram {len(retorno)} retornos")
retorno.head(3)

## Passo 3 · Retorno médio e risco

$$\bar{R} = \frac{1}{n}\sum_{t=1}^{n} R_t \qquad
s^2 = \frac{1}{n-1}\sum_{t=1}^{n}(R_t-\bar{R})^2 \qquad s = \sqrt{s^2}$$

O denominador $n-1$ existe porque a média foi estimada da própria amostra. No `pandas`,
`.std()` já usa $n-1$ por padrão, mas vamos **escrever `ddof=1` mesmo assim**: o código
passa a dizer o que faz, e fica igual ao `STDEV.S` que você usou no Excel. No `numpy` o
padrão é o contrário, e sem `ddof=1` o resultado sai errado.

Anualização: a média multiplica por 252 e o desvio multiplica pela **raiz** de 252.

In [ ]:
# >>> SUA VEZ. Troque cada  ...  pela conta certa.
# >>> As dicas estão nos comentários ao lado.

media_dia = ...      # dica: a média dos retornos diários -> retorno.mean()
desvio_dia = ...     # dica: o desvio dos retornos diários, com ddof=1

media_ano = ...      # dica: a média diária MULTIPLICADA por PREGOES_ANO
desvio_ano = ...     # dica: o desvio diário multiplicado pela RAIZ de PREGOES_ANO
                     #       raiz quadrada em Python: np.sqrt(...)

print(f"{TICKER}")
print(f"  retorno médio ao dia .... {media_dia:.4%}")
print(f"  desvio ao dia ........... {desvio_dia:.3%}")
print(f"  retorno médio ao ano .... {media_ano:.2%}")
print(f"  desvio ao ano ........... {desvio_ano:.2%}")

### Confira contra a planilha

Escreva abaixo os dois números que a **sua planilha** produziu. Se a diferença passar de
0,01 ponto percentual, um dos dois caminhos está errado, e achar qual é parte da tarefa.

In [ ]:
# >>> SUA VEZ. Troque cada  ...  pela conta certa.
# >>> As dicas estão nos comentários ao lado.

EXCEL_RETORNO_ANO = ...    # copie da sua planilha
EXCEL_DESVIO_ANO  = ...

print(f"retorno: Python {media_ano:.2%} × Excel {EXCEL_RETORNO_ANO:.2%}")
print(f"desvio:  Python {desvio_ano:.2%} × Excel {EXCEL_DESVIO_ANO:.2%}")

## Passo 4 · A carteira com o índice

$$\sigma_{12} = \frac{1}{n-1}\sum (R_{1t}-\bar{R_1})(R_{2t}-\bar{R_2})
\qquad \rho = \frac{\sigma_{12}}{\sigma_1\sigma_2}$$

$$\sigma_p^2 = w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1w_2\sigma_{12}$$

Os dois primeiros termos nunca são negativos. Só o terceiro pode ser, e é ele que permite
ao risco da carteira cair abaixo do risco dos dois ativos que a compõem.

In [ ]:
# >>> SUA VEZ. Troque cada  ...  pela conta certa.
# >>> As dicas estão nos comentários ao lado.

par = pd.concat([retorno, retorno_ibov], axis=1).dropna()
par.columns = [TICKER, "BOVA11"]

cov_ano = ...              # dica: par.cov(ddof=1)
corr = ...
desvio_ibov_ano = ...

print(f"covariância ao ano ... {cov_ano:.6f}")
print(f"correlação ........... {corr:.3f}")
print(f"desvio do BOVA11 ..... {desvio_ibov_ano:.2%}")

In [ ]:
# >>> SUA VEZ. Troque cada  ...  pela conta certa.
# >>> As dicas estão nos comentários ao lado.

def risco_carteira(w, s1, s2, cov):
    """Desvio-padrão de uma carteira de dois ativos, com peso w no primeiro."""
    termo1 = ...
    termo2 = ...
    termo3 = ...
    return np.sqrt(termo1 + termo2 + termo3), (termo1, termo2, termo3)

w = 0.5
dp_carteira, termos = risco_carteira(w, desvio_ano, desvio_ibov_ano, cov_ano)

print(f"termo 1 (a sua empresa) .. {termos[0]:.6f}")
print(f"termo 2 (o índice) ....... {termos[1]:.6f}")
print(f"termo 3 (o cruzado) ...... {termos[2]:.6f}")
print(f"desvio da carteira ....... {dp_carteira:.2%}")

## Passo 5 · O desenho

O gráfico abaixo varre todos os pesos possíveis entre 0% e 100% na sua empresa e mostra o
risco de cada carteira. Se a curva tiver barriga, existe uma combinação menos arriscada do
que ficar só no índice.

In [ ]:
# >>> SUA VEZ. Troque cada  ...  pela conta certa.
# >>> As dicas estão nos comentários ao lado.

pesos = np.linspace(0, 1, 201)

# dica: para CADA peso w da lista pesos, chame
#           risco_carteira(w, desvio_ano, desvio_ibov_ano, cov_ano)
#       e guarde só o PRIMEIRO valor devolvido, que é o desvio da carteira.
#       Guardar o primeiro se escreve com [0] no fim da chamada.
#       Tudo isso cabe numa linha, dentro de np.array([ ... for w in pesos ]).
riscos = np.array([...])

# dica: riscos.argmin() devolve a POSIÇÃO do menor risco dentro da lista.
#       Use essa posição para pegar o peso correspondente em pesos.
w_min = ...

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(pesos * 100, riscos * 100, color="#005386", linewidth=2.2)
ax.plot(w_min * 100, riscos.min() * 100, "o", color="#00923E", markersize=7)
ax.annotate(f"risco mínimo: {riscos.min():.1%} com {w_min:.0%} em {TICKER}",
            xy=(w_min * 100, riscos.min() * 100), xytext=(10, 14),
            textcoords="offset points", color="#00923E", fontsize=9)
ax.set_xlabel(f"peso em {TICKER} (%)")
ax.set_ylabel("desvio-padrão da carteira ao ano (%)")
ax.set_title("Risco da carteira conforme o peso", loc="left", fontsize=12)
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

## Passo 6 · A leitura

Responda em texto, no máximo cinco linhas cada, na célula abaixo.

1. O desvio da sua empresa é maior ou menor que o do índice inteiro? Por que isso acontece?
2. A carteira meio a meio ficou menos arriscada que a sua empresa sozinha? O que na fórmula
   explica o resultado?
3. Se a correlação com o índice fosse mais alta, o ganho de juntar as duas seria maior ou
   menor?

*(escreva aqui)*

1.

2.

3.

## Se der erro

Os três erros que quase sempre acontecem, e o que fazer:

**`NameError: name 'media_ano' is not defined`**
Você pulou uma célula, ou a sessão caiu. Faça **Ambiente de execução → Executar tudo**
e depois volte de onde parou.

**`TypeError: unsupported operand type(s) for ...: 'ellipsis'`**
Sobrou um `...` que você não substituiu. O `...` é o buraco a preencher, não é código.
Procure na célula que deu erro.

**A célula não faz nada e fica com um círculo girando**
O Colab está reconectando. Espere alguns segundos. Se não voltar, faça
**Ambiente de execução → Reiniciar sessão** e depois **Executar tudo**.

Nenhum desses erros apaga o seu trabalho. O que você escreveu continua escrito.

## Passo 8 · O seu comentário

Escreva, em poucas linhas, o que os números da sua empresa dizem. Não é resumo do que
você fez: é leitura do resultado. Cite números que só o seu ticker produz.

Escreva **entre as três aspas**, sem apagá-las.

In [ ]:
COMENTARIO = """
(escreva aqui)

Sugestões do que responder, se travar:
- o risco da sua empresa é maior ou menor que o do índice, e por quanto?
- a correlação com o índice foi alta ou baixa? o que isso diz do negócio dela?
- a carteira de peso mínimo ficou com quanto na sua empresa? isso te surpreendeu?
- Python e Excel bateram? se não, onde estava a diferença?
"""

print(f"{len(COMENTARIO.split())} palavras escritas")

## O que entregar

**Dois arquivos, pelo SIGAA, até 16/09:** o arquivo `.md` que a célula abaixo gera e a
**planilha preenchida**.

Os números dos dois têm de bater. Erro honesto e documentado vale mais que resultado
certo sem rastro: o arquivo registra o que você calculou, inclusive o que ficou em
branco, e é assim que a correção enxerga onde a conta parou.

*Dados: pacote congelado da disciplina. Fontes primárias: B3 (preços) e CVM (demonstrações).*

In [ ]:
#@title Gerar o arquivo de entrega (execute por último)
import re, json, unicodedata
from datetime import datetime

def _g(nome):
    """Lê a variável pelo nome. Devolve None se ela nem chegou a existir."""
    return globals().get(nome, None)

def _v(x):
    """Devolve o número, ou None se o aluno não chegou a calcular."""
    try:
        if x is Ellipsis:
            return None
        return float(x)
    except Exception:
        return None

def _pct(x, casas=2):
    v = _v(x)
    return "não calculado" if v is None else f"{v*100:.{casas}f}%".replace(".", ",")

def _num(x, casas=6):
    v = _v(x)
    return "não calculado" if v is None else f"{v:.{casas}f}".replace(".", ",")

def _dif(a, b):
    va, vb = _v(a), _v(b)
    if va is None or vb is None:
        return "não dá para comparar"
    d = f"{(va - vb) * 100:+.3f}".replace(".", ",")
    return f"{d} p.p."

# a data de congelamento vem do próprio nome do arquivo lido
_m = re.search(r"congelado_(\d{4})-(\d{2})-(\d{2})", _g("CAMINHO") or "")
BASE_EM = f"{_m.group(3)}/{_m.group(2)}/{_m.group(1)}" if _m else "não identificada"

_linhas = [
    f"# Risco e retorno · {_g('TICKER') or '?'}",
    "",
    "**ED0139 · Finanças Corporativas · 2026-2 · UFC**",
    "",
    f"- **Aluno:** {_g('NOME') or '(não preenchido)'}",
    f"- **Matrícula:** {_g('MATRICULA') or '(não preenchida)'}",
    f"- **Grupo:** {_g('GRUPO') or '(não preenchido)'}",
    f"- **Empresa:** {_g('TICKER') or '(não definida)'}",
    f"- **Base congelada em:** {BASE_EM}",
    f"- **Gerado em:** {datetime.now():%d/%m/%Y %H:%M}",
    "",
    "## Resultados",
    "",
    "| medida | valor |",
    "|---|---|",
    f"| retorno médio ao dia | {_pct(_g('media_dia'), 4)} |",
    f"| desvio-padrão ao dia | {_pct(_g('desvio_dia'), 3)} |",
    f"| retorno médio ao ano | {_pct(_g('media_ano'))} |",
    f"| desvio-padrão ao ano | {_pct(_g('desvio_ano'))} |",
    f"| desvio-padrão do BOVA11 ao ano | {_pct(_g('desvio_ibov_ano'))} |",
    f"| covariância ao ano | {_num(_g('cov_ano'))} |",
    f"| correlação com o BOVA11 | {_num(_g('corr'), 3)} |",
    f"| desvio da carteira meio a meio | {_pct(_g('dp_carteira'))} |",
    f"| peso em {_g('TICKER')} que minimiza o risco | {_pct(_g('w_min'), 1)} |",
    "",
    "## Python × Excel",
    "",
    "| medida | Python | Excel | diferença |",
    "|---|---|---|---|",
    f"| retorno médio ao ano | {_pct(_g('media_ano'))} | {_pct(_g('EXCEL_RETORNO_ANO'))} | {_dif(_g('media_ano'), _g('EXCEL_RETORNO_ANO'))} |",
    f"| desvio-padrão ao ano | {_pct(_g('desvio_ano'))} | {_pct(_g('EXCEL_DESVIO_ANO'))} | {_dif(_g('desvio_ano'), _g('EXCEL_DESVIO_ANO'))} |",
    "",
    "## Comentário",
    "",
    (_g("COMENTARIO") or "").strip() or "(não escrito)",
    "",
    "---",
    "",
    "```json",
    json.dumps({
        "ticker": _g("TICKER"), "matricula": _g("MATRICULA"), "grupo": _g("GRUPO"),
        "base_congelada_em": BASE_EM, "fonte": _g("CAMINHO"),
        "pregoes": int(len(_g("serie"))) if _g("serie") is not None else None,
        "media_ano": _v(_g("media_ano")), "desvio_ano": _v(_g("desvio_ano")),
        "desvio_ibov_ano": _v(_g("desvio_ibov_ano")), "cov_ano": _v(_g("cov_ano")),
        "corr": _v(_g("corr")), "dp_carteira_meio": _v(_g("dp_carteira")), "w_min": _v(_g("w_min")),
        "excel_retorno_ano": _v(_g("EXCEL_RETORNO_ANO")), "excel_desvio_ano": _v(_g("EXCEL_DESVIO_ANO")),
    }, ensure_ascii=False, indent=2),
    "```",
]
_texto = "\n".join(_linhas)

_id = unicodedata.normalize("NFKD", f"{_g('MATRICULA') or _g('NOME') or 'sem_identificacao'}")
_id = re.sub(r"[^A-Za-z0-9]+", "_", _id).strip("_")[:40] or "sem_identificacao"
ARQUIVO = f"FinCorp_04_{_g('TICKER') or 'sem_ticker'}_{_id}.md"

with open(ARQUIVO, "w", encoding="utf-8") as f:
    f.write(_texto)

_faltando = [c for c in ["NOME", "MATRICULA", "GRUPO"] if not _g(c)]
if _faltando:
    print("ATENÇÃO: falta preencher " + ", ".join(_faltando) + " no Passo 0.\n")
if "não calculado" in _texto:
    print("ATENÇÃO: há medidas não calculadas. O arquivo registra isso, e tudo bem;\n"
          "         mas confira se você não pulou uma célula sem querer.\n")

print(f"Arquivo gerado: {ARQUIVO}\n")
print("-" * 62)
print(_texto[:900])
print("-" * 62)

try:
    from google.colab import files
    files.download(ARQUIVO)
    print("\nO download começou. Se o navegador bloquear, use o painel de arquivos\n"
          "à esquerda (ícone de pasta), clique nos três pontos do arquivo e baixe.")
except Exception:
    print(f"\nO arquivo foi salvo ao lado deste notebook, com o nome acima.")